# 4.5 Fine-tuning

先做 transfer learning，再解凍 MobileNetV2 後段層數微調。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pathlib

## 1. 載入資料

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
data_dir = pathlib.Path(tf.keras.utils.get_file('flower_photos', origin=url, untar=True))
# Some environments extract the archive into an extra nested folder.
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'
img_size = (160, 160)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset='training', seed=42, image_size=img_size, batch_size=batch_size)
val_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset='validation', seed=42, image_size=img_size, batch_size=batch_size)
num_classes = len(train_ds.class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 2. 第一階段：凍結 backbone 訓練分類頭

In [ ]:
tf.keras.utils.set_random_seed(42)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(input_shape=img_size + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=img_size + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_head = model.fit(train_ds, validation_data=val_ds, epochs=3)

## 3. 第二階段：解凍後段層數 fine-tuning

In [ ]:
base_model.trainable = True
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 4. 評估

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print({'val_loss': val_loss, 'val_accuracy': val_acc})

## 5. 套用自己的資料

先確認 transfer learning 已有穩定 baseline，再嘗試 fine-tuning。若 validation loss 上升，代表可能過擬合。

## 6. 小結

Fine-tuning 應建立在穩定的 transfer learning baseline 之後。若 validation loss 上升或 train/validation 差距變大，應減少解凍層數、降低 learning rate 或加強資料增強。
